# Notebook 02: Surgical DRG Cleaning & Transformation
## Purpose
Clean all 4 datasets, filter to surgical DRGs only, calculate
cost metrics, create benchmark tiers, and produce Silver layer
tables ready for dashboard consumption.

## Inputs
- Raw DataFrames from Notebook 01 (reloaded here)

## Outputs
- df_surgical: Filtered surgical procedures with cost metrics
- df_hospital_master: Master hospital reference table
- df_complications_clean: Clean surgical quality measures

## Key Transformations
1. Type casting — string to numeric
2. Surgical DRG filtering
3. Cost metric calculations
4. National benchmark comparisons
5. Cost tier classifications

## Author
Chandra Bajepally
## Date
March 2026

In [0]:
import pandas as pd
import numpy as np

# RELOAD ALL 4 DATASETS

BASE_PATH = "/Volumes/workspace/default/raw_data"

df_provider_service = pd.read_csv(
    f"{BASE_PATH}/cms_inpatient_provider_service.csv",
    dtype=str, encoding='latin-1'
)

df_by_provider = pd.read_csv(
    f"{BASE_PATH}/cms_inpatient_by_provider.csv",
    dtype=str, encoding='latin-1'
)

df_hospital_info = pd.read_csv(
    f"{BASE_PATH}/cms_hospital_general_info.csv",
    dtype=str, encoding='latin-1'
)

df_complications = pd.read_csv(
    f"{BASE_PATH}/cms_complications_deaths.csv",
    dtype=str, encoding='latin-1'
)

print(" All 4 datasets reloaded")
print(f"  Provider & Service: {len(df_provider_service):,} rows")
print(f"  By Provider:        {len(df_by_provider):,} rows")
print(f"  Hospital Info:      {len(df_hospital_info):,} rows")
print(f"  Complications:      {len(df_complications):,} rows")

 All 4 datasets reloaded
  Provider & Service: 146,427 rows
  By Provider:        3,093 rows
  Hospital Info:      5,426 rows
  Complications:      95,780 rows


## Step 1: Type Casting
Convert numeric columns from string to proper numeric types.
We loaded everything as string in Notebook 01 for safety.
Now we cast to correct types for calculations.

In [0]:
# TYPE CASTING — Dataset 1: Provider and Service

numeric_cols_d1 = [
    'Tot_Dschrgs',
    'Avg_Submtd_Cvrd_Chrg',
    'Avg_Tot_Pymt_Amt',
    'Avg_Mdcr_Pymt_Amt'
]

for col in numeric_cols_d1:
    df_provider_service[col] = pd.to_numeric(
        df_provider_service[col], errors='coerce'
    )

# TYPE CASTING — Dataset 2: By Provider

numeric_cols_d2 = [
    'Tot_Benes', 'Tot_Submtd_Cvrd_Chrg', 'Tot_Pymt_Amt',
    'Tot_Mdcr_Pymt_Amt', 'Tot_Dschrgs', 'Tot_Cvrd_Days',
    'Bene_Avg_Age', 'Bene_Avg_Risk_Scre'
]

for col in numeric_cols_d2:
    df_by_provider[col] = pd.to_numeric(
        df_by_provider[col], errors='coerce'
    )

print(" Type casting complete")
print("\nDataset 1 dtypes after casting:")
print(df_provider_service[numeric_cols_d1].dtypes)
print("\nDataset 2 dtypes after casting:")
print(df_by_provider[numeric_cols_d2].dtypes)

 Type casting complete

Dataset 1 dtypes after casting:
Tot_Dschrgs               int64
Avg_Submtd_Cvrd_Chrg    float64
Avg_Tot_Pymt_Amt        float64
Avg_Mdcr_Pymt_Amt       float64
dtype: object

Dataset 2 dtypes after casting:
Tot_Benes                 int64
Tot_Submtd_Cvrd_Chrg      int64
Tot_Pymt_Amt              int64
Tot_Mdcr_Pymt_Amt         int64
Tot_Dschrgs               int64
Tot_Cvrd_Days             int64
Bene_Avg_Age            float64
Bene_Avg_Risk_Scre      float64
dtype: object


## Step 2: Surgical DRG Filtering
CMS data contains 534 DRG codes covering ALL inpatient procedures.
We filter to surgical DRGs only — the procedures directly relevant
to Surgical Services operations.

### Surgical Categories We Keep
- Orthopedic (hip, knee, spine, joint replacement)
- Cardiac Surgery (bypass, valve replacement)
- General Surgery (bowel, gallbladder, hernia)
- Neurosurgery (craniotomy, spinal procedures)
- Vascular Surgery
- Thoracic Surgery

### Why This Matters
Filtering irrelevant DRGs (psychiatric, medical management)
keeps our analysis focused on what Surgical Services actually controls.

In [0]:
# SURGICAL DRG FILTERING
# Filter to DRG codes directly relevant to Surgical Services
# Based on MS-DRG surgical classification categories

# Define surgical DRG ranges by category
# These are the major surgical service lines in any hospital

surgical_drg_ranges = {
    'Cardiac Surgery':      list(range(1, 3)) + list(range(216, 252)),
    'Neurosurgery':         list(range(20, 104)),
    'Orthopedic Surgery':   list(range(453, 520)) + list(range(536, 545)),
    'Spinal Surgery':       list(range(28, 31)) + list(range(453, 460)),
    'Vascular Surgery':     list(range(252, 265)),
    'Thoracic Surgery':     list(range(163, 169)),
    'General Surgery':      list(range(326, 358)) + list(range(370, 395)),
    'Transplant Surgery':   list(range(1, 15)),
}

# Flatten all surgical DRG codes into one list
all_surgical_drgs = []
for category, drg_list in surgical_drg_ranges.items():
    all_surgical_drgs.extend(drg_list)

all_surgical_drgs = list(set(all_surgical_drgs))  # Remove duplicates

# Convert DRG_Cd to integer for comparison
df_provider_service['DRG_Cd_Int'] = pd.to_numeric(
    df_provider_service['DRG_Cd'], errors='coerce'
).astype('Int64')

# Apply the filter
df_surgical = df_provider_service[
    df_provider_service['DRG_Cd_Int'].isin(all_surgical_drgs)
].copy()

# Add surgery category label to each row
def assign_surgery_category(drg_code):
    for category, drg_list in surgical_drg_ranges.items():
        if drg_code in drg_list:
            return category
    return 'Other Surgical'

df_surgical['Surgery_Category'] = df_surgical['DRG_Cd_Int'].apply(
    assign_surgery_category
)

print("=== SURGICAL DRG FILTERING RESULTS ===")
print(f"Original rows:  {len(df_provider_service):,}")
print(f"Surgical rows:  {len(df_surgical):,}")
print(f"Reduction:      {len(df_provider_service) - len(df_surgical):,} rows removed")
print(f"\nSurgical rows by category:")
print(df_surgical['Surgery_Category'].value_counts().to_string())

=== SURGICAL DRG FILTERING RESULTS ===
Original rows:  146,427
Surgical rows:  53,184
Reduction:      93,243 rows removed

Surgical rows by category:
Surgery_Category
General Surgery       15877
Neurosurgery          15271
Orthopedic Surgery    11823
Cardiac Surgery        6639
Vascular Surgery       1570
Thoracic Surgery       1319
Transplant Surgery      685


## Step 3: Cost Metric Calculations
Calculate key financial metrics used in Surgical Services analytics.
These metrics directly feed our Power BI dashboards.

| Metric | Formula | Business Use |
|---|---|---|
| cost_per_discharge | Avg Total Payment / Discharges | Cost efficiency |
| charge_to_payment_ratio | Avg Charge / Avg Payment | Pricing analysis |
| medicare_discount_pct | (Charge - Payment) / Charge | Reimbursement gap |
| volume_weighted_cost | Cost × Discharge Volume | Total cost burden |

In [0]:
# COST METRIC CALCULATIONS

# 1. Charge to Payment Ratio
# How much hospitals charge vs what Medicare actually pays
# Higher ratio = bigger gap between billed and paid
df_surgical['charge_to_payment_ratio'] = (
    df_surgical['Avg_Submtd_Cvrd_Chrg'] /
    df_surgical['Avg_Tot_Pymt_Amt']
).round(2)

# 2. Medicare Discount Percentage
# What % of the charge does Medicare NOT pay
df_surgical['medicare_discount_pct'] = (
    (df_surgical['Avg_Submtd_Cvrd_Chrg'] -
     df_surgical['Avg_Tot_Pymt_Amt']) /
    df_surgical['Avg_Submtd_Cvrd_Chrg'] * 100
).round(2)

# 3. Medicare Payment Rate
# What % of total payment comes from Medicare specifically
df_surgical['medicare_payment_rate'] = (
    df_surgical['Avg_Mdcr_Pymt_Amt'] /
    df_surgical['Avg_Tot_Pymt_Amt'] * 100
).round(2)

# 4. Volume Weighted Cost
# Total cost burden = payment × number of cases
# Shows which procedures drive the most total spend
df_surgical['volume_weighted_cost'] = (
    df_surgical['Avg_Tot_Pymt_Amt'] *
    df_surgical['Tot_Dschrgs']
).round(2)

print(" Cost metrics calculated")
print("\nSample cost metrics for first 5 surgical rows:")
cost_cols = [
    'Rndrng_Prvdr_Org_Name',
    'DRG_Desc',
    'Avg_Submtd_Cvrd_Chrg',
    'Avg_Tot_Pymt_Amt',
    'charge_to_payment_ratio',
    'medicare_discount_pct'
]
print(df_surgical[cost_cols].head().to_string())

 Cost metrics calculated

Sample cost metrics for first 5 surgical rows:
             Rndrng_Prvdr_Org_Name                                                                                  DRG_Desc  Avg_Submtd_Cvrd_Chrg  Avg_Tot_Pymt_Amt  charge_to_payment_ratio  medicare_discount_pct
0  Southeast Health Medical Center  ECMO OR TRACHEOSTOMY WITH MV >96 HOURS OR PRINCIPAL DIAGNOSIS EXCEPT FACE, MOUTH AND NEC          663764.35714     120219.928570                     5.52                  81.89
1  Southeast Health Medical Center  CRANIOTOMY WITH MAJOR DEVICE IMPLANT OR ACUTE COMPLEX CNS PRINCIPAL DIAGNOSIS WITH MCC O          180980.88462      37321.038462                     4.85                  79.38
2  Southeast Health Medical Center  CRANIOTOMY WITH MAJOR DEVICE IMPLANT OR ACUTE COMPLEX CNS PRINCIPAL DIAGNOSIS WITHOUT MC          105824.33333      26936.666667                     3.93                  74.55
3  Southeast Health Medical Center                              CRANIOTOMY 

## Step 4: National Benchmark Calculations
For each surgical DRG, calculate the national average payment.
Then compare each hospital to that national average.
This tells us which hospitals are above/below benchmark —
the core of our Supply Chain Cost dashboard.

In [0]:
# NATIONAL BENCHMARK CALCULATIONS
# For each DRG procedure, calculate national averages
# Then classify each hospital as below/at/above benchmark

# Calculate national average payment per DRG
national_avg = df_surgical.groupby('DRG_Cd').agg(
    national_avg_payment=('Avg_Tot_Pymt_Amt', 'mean'),
    national_avg_charge=('Avg_Submtd_Cvrd_Chrg', 'mean'),
    national_total_discharges=('Tot_Dschrgs', 'sum'),
    hospital_count=('Rndrng_Prvdr_CCN', 'nunique')
).round(2).reset_index()

# Join national averages back to surgical dataframe
df_surgical = df_surgical.merge(
    national_avg,
    on='DRG_Cd',
    how='left'
)

# Calculate variance from national benchmark
df_surgical['payment_vs_national'] = (
    df_surgical['Avg_Tot_Pymt_Amt'] -
    df_surgical['national_avg_payment']
).round(2)

df_surgical['payment_vs_national_pct'] = (
    df_surgical['payment_vs_national'] /
    df_surgical['national_avg_payment'] * 100
).round(2)

# Classify into cost tiers
def cost_tier(pct_diff):
    if pct_diff < -10:
        return 'Below Benchmark'      # More than 10% below average — efficient
    elif pct_diff > 10:
        return 'Above Benchmark'      # More than 10% above average — high cost
    else:
        return 'At Benchmark'         # Within 10% of average — typical

df_surgical['cost_tier'] = df_surgical['payment_vs_national_pct'].apply(
    cost_tier
)

print("=== BENCHMARK CLASSIFICATION RESULTS ===")
print(df_surgical['cost_tier'].value_counts().to_string())
print(f"\nTotal surgical hospital-DRG combinations: {len(df_surgical):,}")

print("\nNational benchmark sample — Top 5 DRGs by volume:")
top_drgs = national_avg.nlargest(5, 'national_total_discharges')[
    ['DRG_Cd', 'national_avg_payment',
     'national_total_discharges', 'hospital_count']
]
print(top_drgs.to_string())

=== BENCHMARK CLASSIFICATION RESULTS ===
cost_tier
Below Benchmark    24118
At Benchmark       15528
Above Benchmark    13538

Total surgical hospital-DRG combinations: 53,184

National benchmark sample — Top 5 DRGs by volume:
    DRG_Cd  national_avg_payment  national_total_discharges  hospital_count
145    392               7499.89                      86037            1951
134    378               8924.72                      78557            1840
164    470              16457.75                      71939            1311
39     065               9482.39                      70626            1774
38     064              17437.18                      63186            1491


In [0]:
# BUILD HOSPITAL MASTER TABLE
# Clean hospital reference table joining info + provider data


# Clean hospital info columns
df_hospital_clean = df_hospital_info[[
    'Facility ID',
    'Facility Name',
    'Address',
    'City/Town',
    'State',
    'ZIP Code',
    'County/Parish',
    'Hospital Type',
    'Hospital Ownership',
    'Emergency Services',
    'Hospital overall rating'
]].copy()

# Rename columns to clean names
df_hospital_clean.columns = [
    'hospital_id',
    'hospital_name',
    'address',
    'city',
    'state',
    'zip_code',
    'county',
    'hospital_type',
    'ownership',
    'emergency_services',
    'overall_rating'
]

# Clean rating — replace 'Not Available' with null
df_hospital_clean['overall_rating'] = df_hospital_clean[
    'overall_rating'
].replace('Not Available', np.nan)

# Convert rating to numeric
df_hospital_clean['overall_rating'] = pd.to_numeric(
    df_hospital_clean['overall_rating'], errors='coerce'
)

print("✅ Hospital master table built")
print(f"Shape: {df_hospital_clean.shape}")
print(f"\nHospital types:")
print(df_hospital_clean['hospital_type'].value_counts().to_string())
print(f"\nRating distribution:")
print(df_hospital_clean['overall_rating'].value_counts(
    dropna=False).sort_index().to_string())

✅ Hospital master table built
Shape: (5426, 11)

Hospital types:
hospital_type
Acute Care Hospitals                    3116
Critical Access Hospitals               1376
Psychiatric                              633
Acute Care - Veterans Administration     132
Childrens                                 94
Rural Emergency Hospital                  39
Acute Care - Department of Defense        32
Long-term                                  4

Rating distribution:
overall_rating
1.0     229
2.0     649
3.0     935
4.0     765
5.0     288
NaN    2560


In [0]:
# CLEAN COMPLICATIONS TABLE
# Filter to surgical quality measures only

# Filter to surgical-relevant measures only
surgical_measures = [
    'COMP_HIP_KNEE',        # Hip/knee replacement complications
    'PSI_90_SAFETY',        # Patient safety composite
    'PSI_03_ULCER',         # Pressure ulcer rate
    'PSI_06_IATRTX',        # Iatrogenic pneumothorax
    'PSI_08_POST_HIP',      # Post-op hip fracture
    'PSI_09_POST_HEM',      # Post-op hemorrhage
    'PSI_11_POST_RESP',     # Post-op respiratory failure
    'PSI_12_POST_PELVIC',   # Post-op sepsis
    'PSI_13_POST_SEPSIS',   # Post-op sepsis
    'PSI_14_POST_DVT',      # Post-op DVT
]

df_complications_clean = df_complications[
    df_complications['Measure ID'].isin(surgical_measures)
].copy()

# Clean score column — convert to numeric
df_complications_clean['Score'] = pd.to_numeric(
    df_complications_clean['Score'], errors='coerce'
)

# Clean compared to national column
df_complications_clean['performance_vs_national'] = \
    df_complications_clean['Compared to National'].str.strip()

print("✅ Complications table cleaned")
print(f"Shape: {df_complications_clean.shape}")
print(f"\nMeasures included:")
print(df_complications_clean['Measure ID'].value_counts().to_string())
print(f"\nPerformance vs National:")
print(df_complications_clean[
    'performance_vs_national'].value_counts().to_string())

✅ Complications table cleaned
Shape: (4789, 19)

Measures included:
Measure ID
COMP_HIP_KNEE    4789

Performance vs National:
performance_vs_national
Not Available                          1682
No Different Than the National Rate    1616
Number of Cases Too Small              1451
Better Than the National Rate            28
Worse Than the National Rate             12


In [0]:
# SAVE SILVER LAYER OUTPUTS

OUTPUT_PATH = "/Volumes/workspace/default/raw_data"

# Save surgical dataset
df_surgical.to_csv(
    f"{OUTPUT_PATH}/silver_surgical_procedures.csv",
    index=False
)

# Save hospital master table
df_hospital_clean.to_csv(
    f"{OUTPUT_PATH}/silver_hospital_master.csv",
    index=False
)

# Save clean complications
df_complications_clean.to_csv(
    f"{OUTPUT_PATH}/silver_complications.csv",
    index=False
)

# Save national benchmarks
national_avg.to_csv(
    f"{OUTPUT_PATH}/silver_national_benchmarks.csv",
    index=False
)

print("Silver layer tables saved successfully")
print(f"\nFiles saved to: {OUTPUT_PATH}")
print(f"  silver_surgical_procedures.csv  — {len(df_surgical):,} rows")
print(f"  silver_hospital_master.csv      — {len(df_hospital_clean):,} rows")
print(f"  silver_complications.csv        — {len(df_complications_clean):,} rows")
print(f"  silver_national_benchmarks.csv  — {len(national_avg):,} rows")

Silver layer tables saved successfully

Files saved to: /Volumes/workspace/default/raw_data
  silver_surgical_procedures.csv  — 53,184 rows
  silver_hospital_master.csv      — 5,426 rows
  silver_complications.csv        — 4,789 rows
  silver_national_benchmarks.csv  — 204 rows


## Notebook 02 Complete — Transformation Summary

### Silver Layer Tables Produced
| Table | Rows | Description |
|---|---|---|
| silver_surgical_procedures | 53,184 | Filtered surgical DRGs with cost metrics |
| silver_hospital_master | 5,426 | Clean hospital reference table |
| silver_complications | 4,789 | Surgical quality measures |
| silver_national_benchmarks | 204 | National average by DRG |

### Key Metrics Added
- charge_to_payment_ratio
- medicare_discount_pct
- medicare_payment_rate
- volume_weighted_cost
- payment_vs_national
- payment_vs_national_pct
- cost_tier (Below/At/Above Benchmark)

In [0]:
# Quick validation summary
print("=== SILVER LAYER VALIDATION ===\n")

print(f"Surgical Procedures Table:")
print(f"  Rows: {len(df_surgical):,}")
print(f"  Columns: {len(df_surgical.columns)}")
print(f"  Surgery Categories: {df_surgical['Surgery_Category'].nunique()}")
print(f"  Unique Hospitals: {df_surgical['Rndrng_Prvdr_CCN'].nunique():,}")
print(f"\n  Cost Tier Distribution:")
print(df_surgical['cost_tier'].value_counts().to_string())

print(f"\nHospital Master Table:")
print(f"  Rows: {len(df_hospital_clean):,}")
print(f"  States covered: {df_hospital_clean['state'].nunique()}")

print(f"\nComplications Table:")
print(f"  Rows: {len(df_complications_clean):,}")
print(f"  Unique hospitals: {df_complications_clean['Facility ID'].nunique():,}")

print(f"\nNational Benchmarks Table:")
print(f"  Rows: {len(national_avg):,}")
print(f"  Avg payment range: ${national_avg['national_avg_payment'].min():,.0f} — ${national_avg['national_avg_payment'].max():,.0f}")

=== SILVER LAYER VALIDATION ===

Surgical Procedures Table:
  Rows: 53,184
  Columns: 28
  Surgery Categories: 7
  Unique Hospitals: 2,464

  Cost Tier Distribution:
cost_tier
Below Benchmark    24118
At Benchmark       15528
Above Benchmark    13538

Hospital Master Table:
  Rows: 5,426
  States covered: 56

Complications Table:
  Rows: 4,789
  Unique hospitals: 4,789

National Benchmarks Table:
  Rows: 204
  Avg payment range: $5,584 — $343,782
